In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
RAW_DIR = "../Datasets/Landmarks"
OUT_DIR = "../Datasets/normalized_landmarks"
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# LANDMARK COUNTS
FACE_LM = 54        # IMPORTANT_FACE_LANDMARKS count
HAND_LM = 21
POSE_LM = 33

In [4]:
TOTAL_LM = FACE_LM + HAND_LM + HAND_LM + POSE_LM
DIM = 3  # x,y,z each

In [5]:
TOTAL_LM

129

In [6]:
# Index helpers (flattened)
FACE_START = 0
LEFT_HAND_START = FACE_LM * DIM
RIGHT_HAND_START = LEFT_HAND_START + HAND_LM * DIM
POSE_START = RIGHT_HAND_START + HAND_LM * DIM

In [7]:
def get_landmark(arr, start_idx, lm_index):
    """Extract a single 3D landmark from flattened vector"""
    base = start_idx + lm_index * 3
    return arr[base:base+3]

In [8]:
def get_landmark(arr, start_idx, lm_index):
    """Extract a single 3D landmark from flattened vector"""
    base = start_idx + lm_index * 3
    return arr[base:base+3]

In [9]:
def compute_center(frame):
    """Center point = pose[0] if available, else mean of both hands"""

    pose0 = get_landmark(frame, POSE_START, 0)

    if np.any(pose0 != 0):
        return pose0

    # fallback: average of wrist positions
    left_wrist = get_landmark(frame, LEFT_HAND_START, 0)
    right_wrist = get_landmark(frame, RIGHT_HAND_START, 0)

    if np.any(left_wrist != 0) or np.any(right_wrist != 0):
        return (left_wrist + right_wrist) / 2

    # final fallback: frame mean
    return np.mean(frame.reshape(-1, 3), axis=0)

In [10]:
def compute_scale(frame):
    """Scale = distance between shoulders (pose 11,12). Fallback = wrist distance."""

    left_shoulder = get_landmark(frame, POSE_START, 11)
    right_shoulder = get_landmark(frame, POSE_START, 12)

    if np.any(left_shoulder != 0) and np.any(right_shoulder != 0):
        return np.linalg.norm(left_shoulder - right_shoulder)

    # fallback: wrist-to-wrist
    lw = get_landmark(frame, LEFT_HAND_START, 0)
    rw = get_landmark(frame, RIGHT_HAND_START, 0)

    if np.any(lw != 0) and np.any(rw != 0):
        return np.linalg.norm(lw - rw)

    # fallback: 1.0 (avoid div by zero)
    return 1.0

In [11]:
def normalize_sequence(seq):
    """Normalize whole video frame-by-frame"""
    norm_seq = []

    for frame in seq:
        center = compute_center(frame)
        scale = compute_scale(frame)

        frame = frame.reshape(-1, 3)
        normalized = (frame - center) / scale
        norm_seq.append(normalized.flatten())

    return np.array(norm_seq)

In [12]:
#     PROCESS ALL WORD FOLDERS    #

views = ["Front"]

for view in views:
    view_raw_path = os.path.join(RAW_DIR, view)
    view_out_path = os.path.join(OUT_DIR, view)
    os.makedirs(view_out_path, exist_ok=True)

    word_folders = os.listdir(view_raw_path)

    for word in tqdm(word_folders, desc=f"Normalizing {view}"):
        raw_word_dir = os.path.join(view_raw_path, word)
        out_word_dir = os.path.join(view_out_path, word)
        os.makedirs(out_word_dir, exist_ok=True)

        files = [f for f in os.listdir(raw_word_dir) if f.endswith(".npy")]

        for f in files:
            arr = np.load(os.path.join(raw_word_dir, f))

            norm = normalize_sequence(arr)

            save_path = os.path.join(out_word_dir, f)
            np.save(save_path, norm)

print("Normalization Completed Successfully!")

Normalizing Front: 100%|██████████| 401/401 [07:54<00:00,  1.18s/it]

Normalization Completed Successfully!
